## Investigating the Invariant Mass vs Missing Mass Mismatch Issue

In [1]:
import pandas as pd
import numpy as np

from spyral.core.constants import AMU_2_MEV, QBRHO_2_P

from spyral_utils.nuclear import NuclearDataMap
from spyral_utils.nuclear.target import GasTarget, load_target, SolidTarget
from spyral_utils.plot import Histogrammer
from pathlib import Path
import vector

In [2]:
nuclear_map = NuclearDataMap()
ic_window_material = SolidTarget(compound=[[1,1,14],[6,12,14],[7,14,4],[8,16,4]], thickness=1422.312, nuclear_data=nuclear_map)
# AT-TPC entrance window, thickness in ug/cm^2
attpc_window_material = SolidTarget(compound=[[1,1,14],[6,12,14],[7,14,4],[8,16,4]], thickness=1422.312, nuclear_data=nuclear_map)
# Ion Chamber gas material, pressure in Torr
# ic_gas_material = GasTarget(compound=[(6,12,1),(9,18,4)], pressure=200.0, nuclear_data=nuclear_map)
# ic_gas_thickness = 0.035 #m

target_material_path = Path("/Users/pranjalsingh/Desktop/research_space_spyral/e20020_analysis/solver_gas_16O.json")

ejectile_z = 2
ejectile_a = 4

# The incoming nucleus (the beam)
projectile_z = 8
projectile_a = 16


# The target nucleus
target_z = 2
target_a = 4

# We calculate the residual for you
residual_z = target_z + projectile_z - ejectile_z
residual_a = target_a + projectile_a - ejectile_a
print(residual_a,residual_z)

if residual_z < 0:
    raise Exception(f"Illegal nuclei! Residual Z: {residual_z}")
if residual_a < 1:
    raise Exception(f"Illegal nuclei! Residual A: {residual_a}")

16 8


In [3]:
target_material = load_target(target_material_path, nuclear_map)
if not isinstance(target_material, GasTarget):
    print('Target error!')

ejectile = nuclear_map.get_data(ejectile_z, ejectile_a)
projectile = nuclear_map.get_data(projectile_z, projectile_a)
# fusion = nuclear_map.get_data(fusion_z, fusion_a)
target = nuclear_map.get_data(target_z, target_a)
residual = nuclear_map.get_data(residual_z, residual_a)
print(f"Reaction: {target}({projectile}, {ejectile}){residual}")
print(f"Target material: {target_material.ugly_string}")

# Initial beam energy
mass_amu = projectile.mass / AMU_2_MEV # If needed, to convert beam energy in MeV/u -> MeV
proj_energy_accel = 161.0 # MeV, the beam energy from the accelerator

# The beam energy after the ic entrance window
proj_energy_ic = proj_energy_accel #- ic_window_material.get_energy_loss(projectile, proj_energy_accel, np.array([0.0]))[0]

# The beam energy after the ic gas
proj_energy_ic_exit = proj_energy_ic #- ic_gas_material.get_energy_loss(projectile, proj_energy_ic, np.array([ic_gas_thickness]))[0]
# The beam energy after the ic exit window
proj_energy_post_ic = proj_energy_ic_exit #- ic_window_material.get_energy_loss(projectile, proj_energy_ic_exit, np.array([0.0]))[0]
# The beam energy after the AT-TPC entrace window
proj_energy_start = proj_energy_post_ic - attpc_window_material.get_energy_loss(projectile, proj_energy_post_ic, np.array([0.0]))[0]
# The beam energy at the downstream end of the AT-TPC
proj_energy_stop = proj_energy_start - target_material.get_energy_loss(projectile, proj_energy_start, np.array([1.0]))[0] # Energy at far end of detector
print(f"Accelerator Beam energy: {proj_energy_accel} MeV")
print(f"Beam energy after IC (2 windows + gas): {proj_energy_post_ic} MeV")
print(f"Beam energy range in AT-TPC: {proj_energy_start}-{proj_energy_stop} MeV")

Reaction: 4He(16O, 4He)16O
Target material: (Gas)4He1
Accelerator Beam energy: 161.0 MeV
Beam energy after IC (2 windows + gas): 161.0 MeV
Beam energy range in AT-TPC: 157.09712267434188-105.57921358612244 MeV


## df from engine

In [4]:
event = 12

In [5]:
df = pd.read_parquet("/Users/pranjalsingh/Desktop/research_space_engine/e20020_engine/my_sim/output_16O_invariant/kinematics/o16a4a_700Torr_161.6MeV_targetalpha.parquet")
df

,event,Z,A,isotope,energy,px,py,pz,vertex_x,vertex_y,vertex_z
0,0,2,4,4He,3727.379333,0.000000,0.000000,0.000000,-0.003115,0.005560,0.250202
1,0,8,16,16O,15045.450442,0.000000,0.000000,2121.827267,-0.003115,0.005560,0.250202
2,0,2,4,4He,3777.478061,67.843404,175.009036,583.740456,-0.003115,0.005560,0.250202
3,0,8,16,16O,14995.351714,-67.843404,-175.009036,1538.086811,-0.003115,0.005560,0.250202
4,1,2,4,4He,3727.379333,0.000000,0.000000,0.000000,-0.012784,-0.000040,0.061096
...,...,...,...,...,...,...,...,...,...,...,...
3995,998,8,16,16O,15007.958878,57.602850,220.775857,1651.404490,0.002490,0.010124,0.392350
3996,999,2,4,4He,3727.379333,0.000000,0.000000,0.000000,0.011941,-0.011265,0.937703
3997,999,8,16,16O,15010.318780,0.000000,0.000000,1856.405625,0.011941,-0.011265,0.937703
3998,999,2,4,4He,3742.619292,-72.659633,-98.574786,314.397940,0.011941,-0.011265,0.937703


In [6]:
filtered_df = df[(df["event"] == event) & (df["Z"]== 2)]

## df from Solver (decay and target)

In [7]:
# run = 0

# """Decay Kinematics
# """
# df_decay = pd.read_parquet(f"/Volumes/researchEXT/O16/all_analysis_spyralv1.0/simulated_16O_decay/InterpSolver/run_{run:04d}_4He.parquet")
# brho_decay = df_decay["brho"]
# momentum_decay = (
#             brho_decay
#             * float(ejectile.Z)
#             * QBRHO_2_P
#         )

# px_decay = momentum_decay * np.sin(df_decay["polar"]) * np.cos(df_decay["azimuthal"])
# py_decay = momentum_decay * np.sin(df_decay["polar"]) * np.sin(df_decay["azimuthal"])
# pz_decay = momentum_decay * np.cos(df_decay["polar"])

# df_decay["px"] = px_decay
# df_decay["py"] = py_decay
# df_decay["pz"] = pz_decay

# energy_decay = np.sqrt(
#             momentum_decay**2.0 + ejectile.mass**2.0
#         )

# df_decay["energy"] = energy_decay


# """Target Kinematics
# """
# df_target = pd.read_parquet(f"/Volumes/researchEXT/O16/all_analysis_spyralv1.0/simulated_16O_target/InterpSolver/run_{run:04d}_4He.parquet")
# brho_target = df_target["brho"]
# momentum_target = (
#             brho_target
#             * float(ejectile.Z)
#             * QBRHO_2_P
#         )

# px_target = momentum_target * np.sin(df_target["polar"]) * np.cos(df_target["azimuthal"])
# py_target = momentum_target * np.sin(df_target["polar"]) * np.sin(df_target["azimuthal"])
# pz_target = momentum_target * np.cos(df_target["polar"])

# df_target["px"] = px_target
# df_target["py"] = py_target
# df_target["pz"] = pz_target

# energy_target = np.sqrt(
#             momentum_target**2.0 + ejectile.mass**2.0
#         )

# df_target["energy"] = energy_target



# filtered_df_decay = df_decay[(df_decay["event"] == event)]
# filtered_df_target = df_target[(df_target["event"] == event)]


In [8]:
filtered_df

,event,Z,A,isotope,energy,px,py,pz,vertex_x,vertex_y,vertex_z
48,12,2,4,4He,3727.379333,0.000000,0.000000,0.000000,-0.000069,-0.002,0.94366
50,12,2,4,4He,3745.311432,51.863515,-119.544243,342.082419,-0.000069,-0.002,0.94366


In [9]:
# filtered_df_decay

In [10]:
# filtered_df_target

## Total energy

### What the sum should be (engine)

In [11]:
# total_inv_engine = filtered_df['energy'].tail(4).sum()
# print(total_inv_engine)

### What is sum is (spyral)

In [12]:
# total_inv_spyral = filtered_df_decay['energy'].tail(4).sum()
# print(total_inv_spyral)

## Invariant energy

### engine

In [13]:
# px_eng_tot = filtered_df["px"].tail(4).sum()
# py_eng_tot = filtered_df["py"].tail(4).sum()
# pz_eng_tot = filtered_df["pz"].tail(4).sum()

# p_eng_tot = np.sqrt(
#             px_eng_tot**2
#             + py_eng_tot**2
#             + pz_eng_tot**2
#         )

# inv_mass_eng = np.sqrt(
#             total_inv_engine**2 - p_eng_tot**2
#         )

# inv_energy_eng = inv_mass_eng - projectile.mass #projectile mass 16O

# print(f"Engine 16O excitation energy: {inv_energy_eng} MeV")


### spyral

In [14]:
# px_spy_tot = filtered_df_decay["px"].tail(4).sum()
# py_spy_tot = filtered_df_decay["py"].tail(4).sum()
# pz_spy_tot = filtered_df_decay["pz"].tail(4).sum()

# p_spy_tot = np.sqrt(
#             px_spy_tot**2
#             + py_spy_tot**2
#             + pz_spy_tot**2
#         )

# inv_mass_spy = np.sqrt(
#             total_inv_spyral**2 - p_spy_tot**2
#         )

# inv_energy_spy = inv_mass_spy - projectile.mass #projectile mass 16O

# print(f"Spyral 16O excitation energy: {inv_energy_spy} MeV")

## Missing mass

### engine

In [15]:
target_vector = vector.array({
    "px": [0.0],
    "py": [0.0],
    "pz": [0.0],
    "E": [target.mass]
})

vertices_eng = filtered_df[['vertex_x', 'vertex_y', 'vertex_z']].to_numpy() 
print(f"Verticies: {vertices_eng}")
distance_eng = np.linalg.norm(vertices_eng, axis=1)

projectile_ke_eng = (proj_energy_start - target_material.get_energy_loss(projectile,proj_energy_start,distance_eng))

projectile_vector_eng = vector.array({
        "px": np.zeros(len(projectile_ke_eng)),
        "py": np.zeros(len(projectile_ke_eng)),
        "pz": np.sqrt(projectile_ke_eng * (projectile_ke_eng + 2.0 * projectile.mass)),
        "E": projectile_ke_eng + projectile.mass
    })


ejectile_vector_eng = vector.array({
    "px": filtered_df["px"].to_numpy(),
    "py": filtered_df["py"].to_numpy(),
    "pz": filtered_df["pz"].to_numpy(),
    "E": filtered_df["energy"].to_numpy()
})
residual_vector = (target_vector + projectile_vector_eng - ejectile_vector_eng)
ex_eng = residual_vector.mass - residual.mass

print(f"Engine 16O excitation energy (missing): {ex_eng[1]} MeV")


Verticies: [[-6.85450057e-05 -1.99995285e-03  9.43659750e-01]
 [-6.85450057e-05 -1.99995285e-03  9.43659750e-01]]
Engine 16O excitation energy (missing): 18.89063032235572 MeV


In [16]:
# target_vector = vector.array({
#     "px": [0.0],
#     "py": [0.0],
#     "pz": [0.0],
#     "E": [target.mass]
# })

# # vertices_eng = filtered_df[['vertex_x', 'vertex_y', 'vertex_z']].to_numpy()
# # print(f"Verticies: {vertices_eng}")
# distance_eng = filtered_df[['vertex_z']].to_numpy()

# projectile_ke_eng = (proj_energy_start - target_material.get_energy_loss(projectile,proj_energy_start,distance_eng))

# projectile_vector_eng = vector.array({
#         "px": np.zeros(len(projectile_ke_eng)),
#         "py": np.zeros(len(projectile_ke_eng)),
#         "pz": np.sqrt(projectile_ke_eng * (projectile_ke_eng + 2.0 * projectile.mass)),
#         "E": projectile_ke_eng + projectile.mass
#     })


# ejectile_vector_eng = vector.array({
#     "px": filtered_df["px"].to_numpy(),
#     "py": filtered_df["py"].to_numpy(),
#     "pz": filtered_df["pz"].to_numpy(),
#     "E": filtered_df["energy"].to_numpy()
# })
# residual_vector = (target_vector + projectile_vector_eng - ejectile_vector_eng)
# ex_eng = residual_vector.mass - residual.mass

# print(f"Engine 16O excitation energy (missing): {ex_eng[1]} MeV")

### spyral

In [17]:
# target_vector = vector.array({
#     "px": [0.0],
#     "py": [0.0],
#     "pz": [0.0],
#     "E": [target.mass]
# })

# vertices_spy = filtered_df_target[['vertex_x', 'vertex_y', 'vertex_z']].to_numpy()
# print(f"Verticies: {vertices_spy}")
# vertices_spy = np.concatenate((vertices_spy,vertices_spy),axis=0) #just did so I can call spyral functions with an array
# distance_spy = np.linalg.norm(vertices_spy, axis=1)

# projectile_ke_spy = (proj_energy_start - target_material.get_energy_loss(projectile,proj_energy_start,distance_spy))

# projectile_vector_spy = vector.array({
#         "px": np.zeros(len(projectile_ke_spy)),
#         "py": np.zeros(len(projectile_ke_spy)),
#         "pz": np.sqrt(projectile_ke_spy * (projectile_ke_spy + 2.0 * projectile.mass)),
#         "E": projectile_ke_spy + projectile.mass
#     })

# brho = filtered_df_target['brho'].to_numpy()
# brho = np.concatenate((brho,brho),axis=0).flatten() #just did so I can call spyral functions with an array
# momentum = brho * float(ejectile.Z) * QBRHO_2_P

# polar = filtered_df_target['polar'].to_numpy()
# polar = np.concatenate((polar,polar),axis=0).flatten()
# az = filtered_df_target['azimuthal'].to_numpy()
# az = np.concatenate((az,az),axis=0).flatten()

# ejectile_vector_spy = vector.array({
#         "px": momentum * np.sin(polar) * np.cos(az),
#         "py": momentum * np.sin(polar) * np.sin(az),
#         "pz": momentum * np.cos(polar),
#         "E": np.sqrt(momentum**2.0 + ejectile.mass**2.0)
#     })



# residual_vector = (target_vector + projectile_vector_spy - ejectile_vector_spy)

# ex_spy = residual_vector.mass - residual.mass

# print(f"Engine 16O excitation energy (missing): {ex_spy[0]} MeV")

### what's the difference between engine and spyral?

#### ejectile vector

In [18]:
# print(ejectile_vector_eng[1])
# print(ejectile_vector_spy[0])

#### target vector is the same for both

#### projectile vector

In [19]:
# print(projectile_vector_eng[1])
# print(projectile_vector_spy[0])

In [20]:
# print(ejectile.mass)